# CBA pipeline: Tornado & Whirlpool

Cost–benefit analysis (CBA) for SSP Uganda modeling. The same workflow runs for two process types:

- **Tornado** — reads from the `tornado` subfolder of the run output, writes `cba_results_ssp_modeling_tornado_{country}.csv`
- **Whirlpool** — reads from the `whirlpool` subfolder, writes `cba_results_ssp_modeling_whirlpool_{country}.csv`

**Parameters (in Setup — Configuration):**
- **COUNTRY** — Drives input file name (`{country}.csv`) and output CSV names.
- **PATH_RUN_OUTPUT** — Path to the run output folder (carpet) that contains `tornado/` and `whirlpool/` subfolders.
- **PATH_CB_OUTPUT** — Path to the folder where result CSVs are written.

**How to run:**  
1. Run the **Setup** cells (imports + config + pipeline function) once.  
2. In the **Run** section, set `RUN_TORNADO` and/or `RUN_WHIRLPOOL` to `True`, then run that cell and the next one.

## Setup — Imports

In [15]:
from costs_benefits_ssp.cb_calculate import CostBenefits
import pathlib
import pandas as pd

## Setup — Configuration

**Parameters you change:**
- **`COUNTRY`** — Country code used for the input CSV name (`{country}.csv`) and for the output CSV name.
- **`PATH_RUN_OUTPUT`** — Path to the run output folder (the “carpet”) that contains the `tornado` and `whirlpool` subfolders. Update when your run ID changes.
- **`PATH_CB_OUTPUT`** — Path to the folder where result CSVs are written.

**Process config** (file naming only): each process uses a subfolder name and an output filename pattern; the actual paths come from the two parameters above.

In [28]:
SSP_PATH = pathlib.Path.cwd().parents[3]
CB_DEFAULT_DEFINITION_PATH = SSP_PATH / "ssp_modeling/cb/cb_cost_factors"
CB_DEFAULT_DEFINITION_FILE_PATH = CB_DEFAULT_DEFINITION_PATH / "cb_config_params.xlsx"
STRATEGY_CODE_BASE = "BASE"

# ---------- Parameters (edit these) ----------
COUNTRY = "uganda"

# Path to the run output folder that contains tornado/ and whirlpool/ subfolders
PATH_RUN_OUTPUT = (
    SSP_PATH / "ssp_modeling/ssp_run_output" /
    "sisepuede_summary_results_run_sisepuede_run_2026-02-18T21;36;42.734194"
)

# Path to the folder where result CSVs are written
PATH_CB_OUTPUT = SSP_PATH / "ssp_modeling/cb/tornado_plot/data/input"

# ---------- Process config (file / subdir names only) ----------
# Subdir under PATH_RUN_OUTPUT; output CSV name under PATH_CB_OUTPUT
PROCESS_CONFIG = {
    "tornado": {
        "run_subdir": "tornado",
        "output_subdir": "tornado",
        "output_csv_name": f"cba_results_ssp_modeling_tornado_{COUNTRY}.csv",
    },
    "whirlpool": {
        "run_subdir": "whirlpool",
        "output_subdir": "whirlpool",
        "output_csv_name": f"cba_results_ssp_modeling_whirlpool_{COUNTRY}.csv",
    },
}

# ---------- Optional: strategy selection per process ----------
# If a value is None -> run ALL strategies.
# If a value is a string -> run only that strategy_code.
# If a value is a list of strings -> run only those strategy_codes.
RUN_STRATEGY_BY_PROCESS = {
    "tornado": ['BASE','AGRC:DEC_CH4_RICE'],     # e.g. "AGRC:DEC_CH4_RICE"
    "whirlpool":  ['BASE','WHIRLPOOL:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ']  # e.g. "AGRC:DEC_CH4_RICE"
}

## Setup — Pipeline function

`run_cba_pipeline(process_name)` runs the full CBA for one process (`"tornado"` or `"whirlpool"`): load data → CostBenefits → system & technical costs → post-process → save CSV.

In [29]:
def run_cba_pipeline(process_name: str) -> None:
    """Run the full CBA pipeline for one process (tornado or whirlpool)."""
    if process_name not in PROCESS_CONFIG:
        raise ValueError(
            f"Unknown process '{process_name}'. Must be one of: {list(PROCESS_CONFIG.keys())}"
        )

    cfg = PROCESS_CONFIG[process_name]
    ssp_run_dir = PATH_RUN_OUTPUT / cfg["run_subdir"]
    cb_output_dir = PATH_CB_OUTPUT / cfg["output_subdir"]
    output_csv_name = cfg["output_csv_name"]

    country_csv = ssp_run_dir / f"{COUNTRY}.csv"
    att_primary_csv = ssp_run_dir / "ATTRIBUTE_PRIMARY.csv"
    att_strategy_csv = ssp_run_dir / "ATTRIBUTE_STRATEGY.csv"

    for p in (country_csv, att_primary_csv, att_strategy_csv):
        if not p.exists():
            raise FileNotFoundError(f"Required input not found: {p}")

    ssp_data = pd.read_csv(country_csv)
    att_primary = pd.read_csv(att_primary_csv)
    att_strategy = pd.read_csv(att_strategy_csv)

    # Optional: filter to one or more strategies for this process.
    # We ALWAYS keep the baseline strategy (STRATEGY_CODE_BASE) so that
    # CostBenefits can compare the chosen strategies against BASE.
    strategy_filter = RUN_STRATEGY_BY_PROCESS.get(process_name)
    if strategy_filter is not None:
        if isinstance(strategy_filter, str):
            chosen_codes = [strategy_filter]
        else:
            chosen_codes = list(strategy_filter)
        # Ensure BASE is always present
        strategy_codes = set(chosen_codes)
        strategy_codes.add(STRATEGY_CODE_BASE)

        before_n = len(att_strategy)
        att_strategy = att_strategy[att_strategy["strategy_code"].isin(strategy_codes)].copy()
        after_n = len(att_strategy)
        if after_n == 0:
            raise ValueError(
                f"No rows in ATTRIBUTE_STRATEGY for strategy_code(s) {strategy_codes} "
                f"for process '{process_name}'. Check RUN_STRATEGY_BY_PROCESS and COUNTRY."
            )
        print(
            f"[{process_name}] Filtering ATTRIBUTE_STRATEGY from {before_n} to {after_n} rows "
            f"for strategy_code(s): {sorted(strategy_codes)} (including BASE)"
        )

    cb = CostBenefits(ssp_data, att_primary, att_strategy, STRATEGY_CODE_BASE)
    cb.load_cb_parameters(str(CB_DEFAULT_DEFINITION_FILE_PATH))

    results_system = cb.compute_system_cost_for_all_strategies()
    results_tx = cb.compute_technical_cost_for_all_strategies()
    results_all = pd.concat([results_system, results_tx], ignore_index=True)
    results_all_pp = cb.cb_process_interactions(results_all)
    results_all_pp_shifted = cb.cb_shift_costs(results_all_pp)

    cb_output_dir.mkdir(parents=True, exist_ok=True)
    output_path = cb_output_dir / output_csv_name
    results_all_pp_shifted.to_csv(output_path, index=False)
    print(f"[{process_name}] Results saved to: {output_path}")

## Run — Choose process(es)

Set to `True` the process(es) you want to run, then run the cell below.

In [30]:
RUN_TORNADO = True   # run tornado CBA
RUN_WHIRLPOOL = True  # run whirlpool CBA

In [31]:
if RUN_TORNADO:
    run_cba_pipeline("tornado")
if RUN_WHIRLPOOL:
    run_cba_pipeline("whirlpool")

# 26 minutes for all runs

[tornado] Filtering ATTRIBUTE_STRATEGY from 64 to 2 rows for strategy_code(s): ['AGRC:DEC_CH4_RICE', 'BASE'] (including BASE)
Cargamos configuración de archivo excel
Se actualizó la base de datos

************************************
*Strategy : AGRC:DEC_CH4_RICE (0/1)
************************************

---------Costs for: cb:wali:technical_cost:sanitation:unimp_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:imp_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:safeman_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:unimp_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:imp_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:safeman_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:omit_rural.
La

/opt/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:660: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_pre2025["variable"] = res_pre2025["variable"] + "_shifted" + (res_pre2025["time_period"]+SSP_GLOBAL_TIME_PERIOD_0).astype(str)#create a new variable so they can be recognized as shifted costs
/opt/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:661: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_pre2025["tim

[whirlpool] Filtering ATTRIBUTE_STRATEGY from 65 to 2 rows for strategy_code(s): ['BASE', 'WHIRLPOOL:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ'] (including BASE)
The TX TX:FRST:INCREASE_SEQUESTRATION_NZ is missing on AttTransformationCode
The TX TX:LNDU:DEC_WETLAND_LOSS_NZ is missing on AttTransformationCode
The TX TX:LNDU:SET_WETLANDS_MINIMUM_NZ is missing on AttTransformationCode
The TX TX:SCOE:INC_EFFICIENCY_HEAT_NZ is missing on AttTransformationCode
Cargamos configuración de archivo excel
Se actualizó la base de datos

************************************
*Strategy : WHIRLPOOL:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ (0/1)
************************************

---------Costs for: cb:wali:technical_cost:sanitation:unimp_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:imp_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:safeman_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:tec

/opt/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:856: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp["difference_variable"] = cb_orm.diff_var
/opt/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:857: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp["difference_value"] = data_merged["difference"]
/opt/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:858: SettingWith

---------Costs for: cb:ippu:technical_cost:abating_gases:other_fgases.
La variable se evalúa en Transformation Cost
---------Costs for: cb:fgtv:technical_cost:flaring:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:fgtv:technical_cost:leaks:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:waso:technical_cost:consumer_food_waste:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:waso:consumer_savings:consumer_food_waste:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:ccsq:technical_cost:direct_air_capture:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:lvst:technical_cost:ent_ferm_mgmt:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:agrc:technical_cost:producer_food_waste:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:agrc:technical_savings:producer_food_waste:X.
La variable se evalúa en Transformation Cost
---------Costs for: cb:a

/opt/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:660: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_pre2025["variable"] = res_pre2025["variable"] + "_shifted" + (res_pre2025["time_period"]+SSP_GLOBAL_TIME_PERIOD_0).astype(str)#create a new variable so they can be recognized as shifted costs
/opt/anaconda3/envs/ssp_uganda_env/lib/python3.11/site-packages/costs_benefits_ssp/cb_calculate.py:661: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_pre2025["tim